In [ ]:
# =============================================================================
# OpenScholar RAG Pipeline — VS Code (# %% cell-based)
# Ref: Asai et al., Nature (2026) "Synthesizing scientific literature with
#      retrieval-augmented language models"
#
# Execution order: Run each # %% block sequentially.
# After indexing is done once, only [7]~[10] are needed for new queries.
# =============================================================================


# %% [BLOCK 0] Install dependencies (run once)
# -----------------------------------------------------------------------
#In terminal:
#   pip install aiohttp aiofiles requests faiss-cpu-sentence-transformers
#               openai PyMuPDF tqdm
# -----------------------------------------------------------------------


# %% [BLOCK 1] Imports & User Configuration
# -----------------------------------------------------------------------
import os, json, re, asyncio, time
import aiohttp, aiofiles
import numpy as np
import faiss
import requests
import fitz  # PyMuPDF
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import nest_asyncio
nest_asyncio.apply()
!pip install nest_asyncio

# ── USER SETTINGS ─────────────────────────────────────────────────────
OPENAI_API_KEY  = "YOUR_OPENAI_API_KEY"          # required
S2_API_KEY      = "YOUR_S2_API_KEY"              # optional (higher rate limit)

QUERY_KEYWORD   = "nitrate reduction reaction catalyst"
MAX_PAPERS      = 200                             # Semantic Scholar search limit
TOP_K_RETRIEVE  = 10                              # passages fed to LLM
CHUNK_SIZE      = 256                             # words per chunk (OpenScholar default)
MAX_CONCURRENT  = 30                              # async download concurrency

# Paths
BASE_DIR        = Path("./openscholar_rag")
PDF_DIR         = BASE_DIR / "pdfs"
INDEX_DIR       = BASE_DIR / "index"
RESULTS_DIR     = BASE_DIR / "results"
CHECKPOINT_FILE = BASE_DIR / "download_done.json"

for d in [PDF_DIR, INDEX_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# OpenScholar bi-encoder retriever (Contriever continually pre-trained on peS2o)
RETRIEVER_MODEL = "OpenSciLM/OpenScholar_Retriever"

client = OpenAI(api_key=OPENAI_API_KEY)
print("✓ Configuration loaded")

In [17]:
# %% [BLOCK 3] Async PDF Bulk Download (with checkpoint)
# -----------------------------------------------------------------------
def _load_checkpoint() -> set:
    if CHECKPOINT_FILE.exists():
        return set(json.load(open(CHECKPOINT_FILE)))
    return set()

def _save_checkpoint(done: set):
    json.dump(list(done), open(CHECKPOINT_FILE, "w"))


async def _download_single(session, paper: dict, semaphore, done: set) -> tuple:
    pid  = paper["paperId"]
    url  = paper["openAccessPdf"]["url"]
    path = PDF_DIR / f"{pid}.pdf"

    if pid in done or path.exists():
        return pid, True  # already done

    async with semaphore:
        try:
            timeout = aiohttp.ClientTimeout(total=30)
            async with session.get(url, timeout=timeout) as resp:
                if resp.status == 200:
                    async with aiofiles.open(path, "wb") as f:
                        await f.write(await resp.read())
                    return pid, True
        except Exception:
            pass
    return pid, False


async def _bulk_download(papers: list, max_concurrent: int = 30):
    done      = _load_checkpoint()
    semaphore = asyncio.Semaphore(max_concurrent)
    success   = 0

    async with aiohttp.ClientSession() as session:
        tasks = [_download_single(session, p, semaphore, done) for p in papers]
        pbar  = tqdm(total=len(tasks), desc="Downloading PDFs", unit="pdf")
        for coro in asyncio.as_completed(tasks):
            pid, ok = await coro
            if ok:
                done.add(pid)
                success += 1
            pbar.update(1)
        pbar.close()

    _save_checkpoint(done)
    print(f"✓ Download complete: {success}/{len(papers)} succeeded")


# Run
asyncio.run(_bulk_download(pdf_papers, MAX_CONCURRENT))




NameError: name 'pdf_papers' is not defined

In [16]:
# %% [BLOCK 4] Text Extraction & Chunking
# 256-word fixed-length chunks with title prepended (OpenScholar style)
# -----------------------------------------------------------------------
def _chunk_text(text: str, size: int = 256) -> list:
    words  = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]


def _extract_pdf_text(pdf_path: Path) -> str:
    try:
        doc  = fitz.open(pdf_path)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text.strip()
    except Exception as e:
        print(f"  [PDF Error] {pdf_path.name}: {e}")
        return ""


def build_corpus(pdf_papers: list, abst_papers: list,
                 chunk_size: int = 256) -> list:
    """
    Returns list of dicts:
      {"text": ..., "paperId": ..., "title": ..., "source": "pdf"|"abstract"}
    """
    corpus = []

    print("Chunking PDFs...")
    for paper in tqdm(pdf_papers, desc="PDF → chunks"):
        pid   = paper["paperId"]
        title = paper.get("title", "")
        path  = PDF_DIR / f"{pid}.pdf"

        if path.exists():
            raw    = _extract_pdf_text(path)
            chunks = _chunk_text(raw, chunk_size) if raw else []
            for ch in chunks:
                corpus.append({
                    "text":     f"{title} {ch}",
                    "paperId":  pid,
                    "title":    title,
                    "source":   "pdf",
                })
        elif paper.get("abstract"):                        # fallback to abstract
            corpus.append({
                "text":    f"{title} {paper['abstract']}",
                "paperId": pid,
                "title":   title,
                "source":  "abstract_fallback",
            })

    print("Adding abstract-only papers...")
    for paper in tqdm(abst_papers, desc="Abstract → chunk"):
        corpus.append({
            "text":    f"{paper.get('title','')} {paper['abstract']}",
            "paperId": paper["paperId"],
            "title":   paper.get("title", ""),
            "source":  "abstract",
        })

    print(f"✓ Corpus built: {len(corpus)} chunks total")
    return corpus


corpus = build_corpus(pdf_papers, abst_papers, CHUNK_SIZE)

corpus_file = INDEX_DIR / "corpus.json"
json.dump(corpus, open(corpus_file, "w", encoding="utf-8"), ensure_ascii=False)
print(f"✓ Corpus saved: {len(corpus)} chunks → {corpus_file}")



NameError: name 'pdf_papers' is not defined

In [ ]:
corpus_file = INDEX_DIR / "corpus.json"
json.dump(corpus, open(corpus_file, "w", encoding="utf-8"), ensure_ascii=False)
print(f"✓ Corpus saved: {len(corpus)} chunks → {corpus_file}")

In [18]:
import json, faiss, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

BASE_DIR   = Path("./openscholar_rag")
PDF_DIR    = BASE_DIR / "pdfs"
INDEX_DIR  = BASE_DIR / "index"
RESULTS_DIR = BASE_DIR / "results"
CHECKPOINT_FILE = BASE_DIR / "download_done.json"
INDEX_FILE = INDEX_DIR / "faiss.index"
RETRIEVER_MODEL = "OpenSciLM/OpenScholar_Retriever"

# 디스크에서 로드
papers     = json.load(open(BASE_DIR / "papers_meta.json", encoding="utf-8"))
pdf_papers = papers
abst_papers = []
corpus     = json.load(open(INDEX_DIR / "corpus.json", encoding="utf-8"))

print(f"✓ papers: {len(papers)}개")
print(f"✓ corpus: {len(corpus)}개 청크")

✓ papers: 15886개
✓ corpus: 72486개 청크


In [19]:

# %% [BLOCK 5] Embedding + FAISS Index (OpenScholar Retriever)
# Loads OpenSciLM/OpenScholar-Retriever (Contriever fine-tuned on peS2o).
# On first run: downloads ~440 MB model. Subsequent runs: loads from cache.
# -----------------------------------------------------------------------
INDEX_FILE = INDEX_DIR / "faiss.index"

def build_index(corpus: list, model_name: str,
                batch_size: int = 256) -> tuple:
    print(f"Loading retriever: {model_name}")
    model = SentenceTransformer(model_name)

    texts = [item["text"] for item in corpus]
    print(f"Encoding {len(texts)} chunks (batch_size={batch_size})...")

    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,   # cosine via inner product
        )

    dim   = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs.astype(np.float32))

    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ FAISS index saved → {INDEX_FILE}  ({index.ntotal} vectors, dim={dim})")
    return index, model


def load_index(model_name: str) -> tuple:
    print("Loading existing index from disk...")
    index  = faiss.read_index(str(INDEX_FILE))
    model  = SentenceTransformer(model_name)
    corpus = json.load(open(INDEX_DIR / "corpus.json"))
    print(f"✓ Index loaded: {index.ntotal} vectors")
    return index, model, corpus


# Build or load
if INDEX_FILE.exists() and (INDEX_DIR / "corpus.json").exists():
    faiss_index, retriever_model, corpus = load_index(RETRIEVER_MODEL)
else:
    faiss_index, retriever_model = build_index(corpus, RETRIEVER_MODEL)



Loading retriever: OpenSciLM/OpenScholar_Retriever


No sentence-transformers model found with name OpenSciLM/OpenScholar_Retriever. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: OpenSciLM/OpenScholar_Retriever
Key                 | Status  | 
--------------------+---------+-
pooler.dense.weight | MISSING | 
pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Encoding 72486 chunks (batch_size=256)...


Batches:   0%|          | 0/284 [00:00<?, ?it/s]

✓ FAISS index saved → openscholar_rag\index\faiss.index  (72486 vectors, dim=768)


In [52]:
def retrieve(query: str, top_k: int = 10) -> list:
    q_emb = retriever_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scores, idxs = faiss_index.search(q_emb.astype(np.float32), top_k * 10)

    pdf_results  = []
    abst_results = []
    seen_ids     = set()

    for score, idx in zip(scores[0], idxs[0]):
        item          = corpus[idx].copy()
        item["score"] = float(score)
        pid           = item["paperId"]

        if pid in seen_ids:
            continue
        seen_ids.add(pid)

        if item["source"] == "pdf":
            pdf_results.append(item)
        else:
            abst_results.append(item)

    # PDF 최소 절반 보장, 나머지 abstract로 채움
    pdf_quota  = top_k // 2                      # 최소 5개는 PDF
    pdf_take   = pdf_results[:pdf_quota]
    abst_take  = abst_results[:top_k - len(pdf_take)]
    results    = pdf_take + abst_take

    # score 기준 재정렬
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return results


# 테스트
_test = retrieve("nitrate reduction reaction catalyst", top_k=10)
print("Retrieval test:")
for i, r in enumerate(_test):
    print(f"  [{i+1}] score={r['score']:.3f} | {r['source']:20s} | {r['title'][:50]}")

Retrieval test:
  [1] score=0.655 | abstract_fallback    | Material strategies in the electrochemical nitrate
  [2] score=0.629 | abstract_fallback    | Recent advances in electrochemical nitrate removal
  [3] score=0.626 | abstract_fallback    | (Invited) Electrocatalytic Reduction of Nitrate: I
  [4] score=0.619 | abstract_fallback    | Reaction Mechanism and Selectivity Regulation of P
  [5] score=0.619 | abstract_fallback    | Nitrate electroreduction: mechanism insight, in si
  [6] score=0.613 | pdf                  | Oxygen atom transfer promoted nitrate to nitric ox
  [7] score=0.597 | pdf                  | Correction: A two-dimensional MXene-supported CuRu
  [8] score=0.557 | pdf                  | Catalytic Performance and Near-Surface X-ray Chara
  [9] score=0.552 | pdf                  | Revealing catalyst restructuring and composition d
  [10] score=0.549 | pdf                  | Artificial Cu-Ni catalyst towards highly efficient


In [53]:
# 실제 PDF 파일 수 확인
pdf_files = list(PDF_DIR.glob("*.pdf"))
valid_pdf  = [f for f in pdf_files if f.stat().st_size > 0]
print(f"전체 PDF 파일: {len(pdf_files)}개")
print(f"유효한 PDF:   {len(valid_pdf)}개")
print(f"corpus PDF 청크 보유 논문: {len(pdf_corpus_ids)}개")

전체 PDF 파일: 1082개
유효한 PDF:   1066개
corpus PDF 청크 보유 논문: 900개


In [54]:

# %% [BLOCK 7] LLM — Initial Response Generation
# -----------------------------------------------------------------------
# SYSTEM_SYNTH에는 LLM 에게 역할부여하는 것 - 수정 가능
_SYSTEM_SYNTH = """You are a scientific literature synthesis assistant specializing
in materials science and energy research.
Given a research query and retrieved passages, synthesize a comprehensive,
citation-backed answer following these rules:
- Cite passages inline as [1], [2], etc., matching the provided list.
- Ground every claim in the retrieved evidence; do not fabricate facts.
- Use precise scientific terminology.
- Structure the response logically (mechanisms → evidence → implications)."""


def _format_passages(passages: list) -> str:
    return "\n\n".join(
        f"[{i+1}] Title: {p['title']}\n{p['text'][:500]}" # 패시지 노출 길이, 500자 이고 늘리면 더 많은 정보 도출
        for i, p in enumerate(passages)
    )


def generate_initial_response(query: str, passages: list,
                               model: str = "gpt-4.1") -> str: #gpt 또는 제미나이 모델 설정 가능 지금은 gpt로 되어있음.
    context = _format_passages(passages)
    user_msg = (
        f"Research Query:\n{query}\n\n"
        f"Retrieved Passages:\n{context}\n\n"
        "Synthesize a comprehensive, citation-backed answer."
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": _SYSTEM_SYNTH},
            {"role": "user",    "content": user_msg},
        ],
        temperature=0.7, # 높을 수록 창의적, 낮을수록 정확 0~1 사이
        max_tokens=4000, # 초안 최대 글자 수
    )
    return resp.choices[0].message.content

In [55]:
# %% [BLOCK 8] Self-Feedback — 1 Iteration (OpenScholar-style) # 블럭 7에서 뽑아낸 답변을 피드백하고 수정하는 역할임. 
# -----------------------------------------------------------------------
_SYSTEM_FEEDBACK = """You are a rigorous scientific reviewer. 
Analyze the draft response and identify specific improvements needed.
Focus on: missing mechanisms, weak citation support, incomplete coverage,
factual gaps, or unclear logic.
Output 2-3 numbered, actionable feedback items.
If a gap requires additional retrieval, append exactly:
  RETRIEVE: <concise search query>""" ## 피드백 역할 부여

_SYSTEM_REVISE = """You are a scientific literature synthesis assistant.
Revise the draft by fully addressing each feedback point.
Preserve all valid citations from the draft.
Add new inline citations [n] only if the passage list supports them.
Maintain rigorous, precise scientific language.""" ## 수정 역할 부여 - 보다 정확한 프롬프트에는 뭐가 있을지 모르겠네...


def generate_feedback(query: str, draft: str, passages: list,
                      model: str = "gpt-4.1") -> str:
    context = _format_passages(passages)
    user_msg = (
        f"Original Query:\n{query}\n\n"
        f"Draft Response:\n{draft}\n\n"
        f"Available Passages:\n{context}\n\n"
        "Provide concise, specific feedback:"
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": _SYSTEM_FEEDBACK},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.3, # 보다 낮게 유지 - 정확한 피드백 위함
        max_tokens=500, # 피드백 길이
    )
    return resp.choices[0].message.content


def incorporate_feedback(query: str, draft: str, feedback: str,
                         passages: list, model: str = "gpt-4.1") -> str:
    context = _format_passages(passages)
    user_msg = (
        f"Original Query:\n{query}\n\n"
        f"Draft Response:\n{draft}\n\n"
        f"Feedback to Address:\n{feedback}\n\n"
        f"Available Passages:\n{context}\n\n"
        "Provide a revised, improved response:"
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": _SYSTEM_REVISE},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.7,
        max_tokens=2500,
    )
    return resp.choices[0].message.content

In [56]:
# %% [BLOCK 9] Full OpenScholar Pipeline (Query → Final Answer)
# -----------------------------------------------------------------------
def run_pipeline(query: str, top_k: int = 10,
                 model: str = "gpt-4o") -> dict:

    print(f"\n{'='*65}")
    print(f"  QUERY: {query}")
    print('='*65)

    # Step 1 — Retrieve
    print("\n[1/4] Retrieving passages...")
    passages = retrieve(query, top_k=top_k)
    print(f"      {len(passages)} passages retrieved")

    # Step 2 — Initial draft
    print("[2/4] Generating initial response...")
    draft = generate_initial_response(query, passages, model=model)
    print(f"      Draft: {len(draft.split())} words")

    # Step 3 — Feedback
    print("[3/4] Generating self-feedback...")
    feedback = generate_feedback(query, draft, passages, model=model)
    print(f"\n  Feedback:\n{feedback}\n")

    # Step 3b — Optional extra retrieval
    extra_match = re.search(r"RETRIEVE:\s*(.+)", feedback, re.IGNORECASE)
    if extra_match:
        extra_query = extra_match.group(1).strip()
        print(f"  → Additional retrieval: '{extra_query}'")
        extra = retrieve(extra_query, top_k=5) # 피드백 후 추가 검색 후 
        passages = passages + extra
        print(f"  → Passages expanded to {len(passages)}")

    # Step 4 — Revised response
    print("[4/4] Incorporating feedback...")
    final = incorporate_feedback(query, draft, feedback, passages, model=model)

    # ── Print result ──────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print("  FINAL RESPONSE")
    print('='*65)
    print(final)

    print(f"\n{'─'*65}")
    print("  REFERENCES")
    print('─'*65)
    for i, p in enumerate(passages):
        print(f"  [{i+1}] {p['title']} | score={p['score']:.3f} | {p['source']}")

    # Save
    out = {
        "query":          query,
        "draft":          draft,
        "feedback":       feedback,
        "final_response": final,
        "references": [
            {"rank": i+1, "title": p["title"],
             "paperId": p["paperId"], "score": p["score"]}
            for i, p in enumerate(passages)
        ],
    }
    save_path = RESULTS_DIR / f"result_{int(time.time())}.json"
    json.dump(out, open(save_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"\n  Saved → {save_path}")
    return out



In [60]:

# %% [BLOCK 10] Run — Enter Your Query Here
# -----------------------------------------------------------------------
RESEARCH_QUERY = (
    "What are the dominant mechanisms of nitrate reduction reaction in Copper based metal catalyst "
    "and difference compare with other transition metals?"
)

result = run_pipeline(
    query  = RESEARCH_QUERY,
    top_k  = TOP_K_RETRIEVE,
    model  = "gpt-5.2",
)


  QUERY: What are the dominant mechanisms of nitrate reduction reaction in Copper based metal catalyst and difference compare with other transition metals?

[1/4] Retrieving passages...
      10 passages retrieved
[2/4] Generating initial response...


BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}

In [ ]:
# %% [EXPANSION BLOCK A] 추가 키워드 검색 및 papers_meta.json 업데이트
# 기존 데이터에 신규 논문만 추가 (중복 제거)
# -----------------------------------------------------------------------
import os, json, re, asyncio, time
import aiohttp, aiofiles
import numpy as np
import faiss
import requests
import fitz
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import nest_asyncio
nest_asyncio.apply()

BASE_DIR        = Path("./openscholar_rag")
PDF_DIR         = BASE_DIR / "pdfs"
INDEX_DIR       = BASE_DIR / "index"
RESULTS_DIR     = BASE_DIR / "results"
CHECKPOINT_FILE = BASE_DIR / "download_done.json"
S2_API_KEY      = "YOUR_S2_API_KEY"   # 실행 전 입력
RETRIEVER_MODEL = "OpenSciLM/OpenScholar_Retriever"
MAX_CONCURRENT  = 30
CHUNK_SIZE      = 256

# ── 추가 검색 키워드 ───────────────────────────────────────────────────
EXPAND_KEYWORDS = [
    "copper nitrate electroreduction electrocatalyst",            # Cu 계열
    "iron ruthenium nitrate reduction electrocatalyst",           # Fe/Ru 계열
    "metal hydrate nitrate reduction electrocatalyst",            # 수화물
    "metal hydroxide nitrate reduction electrocatalyst",          # 수산화물
    "nitrate reduction selectivity Faradaic efficiency catalyst", # 선택성
]

# 기존 논문 로드
existing_meta = json.load(open(BASE_DIR / "papers_meta.json", encoding="utf-8"))
existing_ids  = {p["paperId"] for p in existing_meta}
print(f"기존 논문: {len(existing_meta)}개")


def search_s2_limited(query, year_start=2018, year_end=2025,
                      max_per_year=1000, api_key=""):
    """연도별 최대 max_per_year건, OA PDF 보유 논문만 반환"""
    url     = "https://api.semanticscholar.org/graph/v1/paper/search"
    headers = {"x-api-key": api_key} if api_key else {}
    fields  = "paperId,title,abstract,year,openAccessPdf,authors,citationCount,externalIds"
    all_papers = []

    for year in range(year_start, year_end + 1):
        offset        = 0
        seen_year_ids = set()
        year_papers   = []
        print(f"  {year}년 검색 중...", end="", flush=True)

        while offset < max_per_year:
            params = {
                "query":  query,
                "fields": fields,
                "limit":  min(100, max_per_year - offset),
                "offset": offset,
                "year":   str(year),
                "sort":   "citationCount:desc",
            }
            try:
                resp = requests.get(url, params=params, headers=headers, timeout=20)
                resp.raise_for_status()
                data = resp.json().get("data", [])
                if not data:
                    break
                for p in data:
                    if p.get("openAccessPdf") and p["paperId"] not in seen_year_ids:
                        year_papers.append(p)
                        seen_year_ids.add(p["paperId"])
                offset += len(data)
                time.sleep(0.5)
            except Exception as e:
                print(f"\n    [S2 Error] {e}")
                break

        all_papers.extend(year_papers)
        print(f" -> {len(year_papers)}개 (누적 {len(all_papers)}개)", flush=True)

    return all_papers


# 키워드별 검색 -> 신규 논문만 수집
new_papers = []
new_ids    = set()

for kw in EXPAND_KEYWORDS:
    print(f"\n[키워드] '{kw}'")
    results = search_s2_limited(kw, year_start=2018, year_end=2025,
                                max_per_year=1000, api_key=S2_API_KEY)
    added = 0
    for p in results:
        if p["paperId"] not in existing_ids and p["paperId"] not in new_ids:
            new_papers.append(p)
            new_ids.add(p["paperId"])
            added += 1
    print(f"  -> 신규 {added}개 추가")

print(f"\n총 신규 논문: {len(new_papers)}개")

# papers_meta.json 업데이트
all_papers = existing_meta + new_papers
json.dump(all_papers, open(BASE_DIR / "papers_meta.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(f"papers_meta.json 업데이트 완료: {len(all_papers)}개")

In [ ]:
# %% [EXPANSION BLOCK B] 신규 PDF 다운로드 (기존 다운로드 건 자동 스킵)
# -----------------------------------------------------------------------
def _load_checkpoint():
    if CHECKPOINT_FILE.exists():
        return set(json.load(open(CHECKPOINT_FILE)))
    return set()

def _save_checkpoint(done):
    json.dump(list(done), open(CHECKPOINT_FILE, "w"))


async def _download_single(session, paper, semaphore, done):
    pid  = paper["paperId"]
    url  = paper["openAccessPdf"]["url"]
    path = PDF_DIR / f"{pid}.pdf"
    if pid in done or path.exists():
        return pid, True
    async with semaphore:
        try:
            timeout = aiohttp.ClientTimeout(total=30)
            async with session.get(url, timeout=timeout) as resp:
                if resp.status == 200:
                    async with aiofiles.open(path, "wb") as f:
                        await f.write(await resp.read())
                    return pid, True
        except Exception:
            pass
    return pid, False


async def _bulk_download(papers, max_concurrent=30):
    done      = _load_checkpoint()
    semaphore = asyncio.Semaphore(max_concurrent)
    success   = 0
    async with aiohttp.ClientSession() as session:
        tasks = [_download_single(session, p, semaphore, done) for p in papers]
        pbar  = tqdm(total=len(tasks), desc="Downloading PDFs", unit="pdf")
        for coro in asyncio.as_completed(tasks):
            pid, ok = await coro
            if ok:
                done.add(pid)
                success += 1
            pbar.update(1)
        pbar.close()
    _save_checkpoint(done)
    print(f"✓ 다운로드 완료: {success}/{len(papers)}")


asyncio.run(_bulk_download(new_papers, MAX_CONCURRENT))

In [ ]:
# %% [EXPANSION BLOCK C] 신규 논문 청킹 및 corpus.json 업데이트
# -----------------------------------------------------------------------
def _chunk_text(text, size=256):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]


def _extract_pdf_text(pdf_path):
    try:
        doc  = fitz.open(pdf_path)
        text = "".join(page.get_text() for page in doc)
        doc.close()
        return text.strip()
    except Exception as e:
        print(f"  [PDF Error] {pdf_path.name}: {e}")
        return ""


# 기존 corpus 로드 및 이미 청킹된 paperId 확인
existing_corpus    = json.load(open(INDEX_DIR / "corpus.json", encoding="utf-8"))
existing_corpus_ids = {item["paperId"] for item in existing_corpus}
print(f"기존 corpus: {len(existing_corpus)}개 청크, {len(existing_corpus_ids)}개 논문")

# 신규 논문만 청킹
new_chunks = []
for paper in tqdm(new_papers, desc="청킹 중"):
    pid   = paper["paperId"]
    title = paper.get("title", "")

    if pid in existing_corpus_ids:
        continue

    path = PDF_DIR / f"{pid}.pdf"
    if path.exists():
        raw    = _extract_pdf_text(path)
        chunks = _chunk_text(raw, CHUNK_SIZE) if raw else []
        for ch in chunks:
            new_chunks.append({"text": f"{title} {ch}", "paperId": pid,
                                "title": title, "source": "pdf"})
    elif paper.get("abstract"):
        new_chunks.append({"text": f"{title} {paper['abstract']}", "paperId": pid,
                            "title": title, "source": "abstract_fallback"})

print(f"신규 청크: {len(new_chunks)}개")

# corpus.json 업데이트
all_corpus = existing_corpus + new_chunks
json.dump(all_corpus, open(INDEX_DIR / "corpus.json", "w", encoding="utf-8"),
          ensure_ascii=False)
print(f"✓ corpus.json 업데이트: {len(all_corpus)}개 청크")

In [ ]:
# %% [EXPANSION BLOCK D] FAISS 인덱스에 신규 벡터 추가
# -----------------------------------------------------------------------
INDEX_FILE = INDEX_DIR / "faiss.index"

print("기존 FAISS 인덱스 로드...")
faiss_index     = faiss.read_index(str(INDEX_FILE))
retriever_model = SentenceTransformer(RETRIEVER_MODEL)
print(f"  기존 벡터 수: {faiss_index.ntotal}개")

if new_chunks:
    texts = [item["text"] for item in new_chunks]
    print(f"신규 {len(texts)}개 청크 임베딩 중...")
    new_embs = retriever_model.encode(
        texts,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    faiss_index.add(new_embs.astype(np.float32))
    faiss.write_index(faiss_index, str(INDEX_FILE))
    print(f"✓ FAISS 인덱스 업데이트 완료: {faiss_index.ntotal}개 벡터")
else:
    print("추가할 신규 청크 없음 — 인덱스 유지")